# 02 — LangChain Introduction

Goal: Understand LangChain basics and rebuild the ChatBot using LangChain components.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

# Create the LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY")
)

print("✅ LangChain LLM ready")

✅ LangChain LLM ready


In [2]:
# Simple invocation
response = llm.invoke("What is the capital of France?")

print("Type:", type(response))
print("Content:", response.content)

Type: <class 'langchain_core.messages.ai.AIMessage'>
Content: The capital of France is Paris.


In [6]:
from langchain_core.messages import SystemMessage,HumanMessage

response = llm.invoke([
    SystemMessage(content="You are a SQL expert. Answer in one short sentence"),
    HumanMessage(content="What is primary key")
])

print(response.content)

A primary key is a unique identifier for each record in a database table.


In [8]:
from langchain_core.messages import SystemMessage , HumanMessage, AIMessage
messages =[
    SystemMessage(content="You are a friendly assistant. Keep answers short.")

]
# First turn
messages.append(HumanMessage(content="My name is Abstract"))
response=llm.invoke(messages)
messages.append(response)
print("Bot:", response.content)
# Second turn — bot should remember
messages.append(HumanMessage(content="What is my name?"))
response = llm.invoke(messages)
messages.append(response)
print("Bot:", response.content)

# Show full history
print("\n--- Full history ---")
for msg in messages:
    print(f"[{msg.__class__.__name__}]: {msg.content}")



Bot: Nice to meet you, Abstract! How can I assist you today?
Bot: Your name is Abstract.

--- Full history ---
[SystemMessage]: You are a friendly assistant. Keep answers short.
[HumanMessage]: My name is Abstract
[AIMessage]: Nice to meet you, Abstract! How can I assist you today?
[HumanMessage]: What is my name?
[AIMessage]: Your name is Abstract.


In [9]:
# Quick exercise: build a chat function using LangChain messages

def simple_chat():
    messages = [
        SystemMessage(content="You are a helpful assistant. Keep answers short.")
    ]
    
    print("Type 'quit' to exit.\n")
    
    while True:
        user_input = input("You: ")
        if user_input.lower() == "quit":
            break
        
        messages.append(HumanMessage(content=user_input))
        response = llm.invoke(messages)
        messages.append(response)
        print(f"Bot: {response.content}\n")

# Run it
simple_chat()

Type 'quit' to exit.

Bot: Hello! How can I assist you today?

Bot: 2 + 2 = 4

Bot: Both the 1kg stone and 1kg cotton weigh the same - 1kg.

Bot: Asus laptops are known for their quality, performance, and affordability. They offer a wide range of laptops for different needs, from everyday use to gaming and professional work.



In [10]:
response = llm.invoke("What is 5847 × 9382? Just give the number, no explanation.")
print("GPT-3.5 answer:", response.content)

# Real answer
print("Actual:", 5847 * 9382)

GPT-3.5 answer: 54857354
Actual: 54856554


### Working with tools

In [ ]:
from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b

# Test the tool directly (without LLM)
result = multiply.invoke({"a": 5847, "b": 9382})
print("Direct call:", result)

Direct call: 54856554


In [12]:
# Bind the tool to the LLM
llm_with_tools = llm.bind_tools([multiply])

# Ask LLM a math question
response = llm_with_tools.invoke("What is 5847 multiplied by 9382?")

print("Response content:", response.content)
print("\nTool calls:", response.tool_calls)

Response content: 

Tool calls: [{'name': 'multiply', 'args': {'a': 5847, 'b': 9382}, 'id': 'call_axLZcQ7RP4pzJ68K3Zus5Qkv', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 9382, 'b': 5847}, 'id': 'call_JoY2fXmeH7e1beZRlPlRlgqf', 'type': 'tool_call'}]


In [13]:
from langchain_core.messages import SystemMessage, HumanMessage

response = llm_with_tools.invoke([
    SystemMessage(content="Be efficient. Use each tool only once per task."),
    HumanMessage(content="What is 5847 multiplied by 9382?")
])

print("Tool calls:", response.tool_calls)

Tool calls: [{'name': 'multiply', 'args': {'a': 5847, 'b': 9382}, 'id': 'call_HR7DooOtWKW5PikY6D0xGieu', 'type': 'tool_call'}]


In [15]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

# Step 1: Initial messages
messages = [
    SystemMessage(content="Be efficient. Use each tool only once per task."),
    HumanMessage(content="What is 5847 multiplied by 9382?")
]

# Step 2: LLM decides what to do
response = llm_with_tools.invoke(messages)
messages.append(response)  # add LLM's decision to history

print("Step 2 — LLM decision:")
print("  Content:", response.content)
print("  Tool calls:", response.tool_calls)
print()

# Step 3: Execute the tool calls
for tool_call in response.tool_calls:
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    tool_id = tool_call["id"]
    
    # Call the tool
    result = multiply.invoke(tool_args)
    print(f"Step 3 — Tool '{tool_name}' executed: {tool_args} = {result}")
    
    # Add tool result to history
    messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))

print()

# Step 4: LLM sees the result and gives final answer
final_response = llm_with_tools.invoke(messages)
print("Step 4 — Final answer:", final_response.content)

Step 2 — LLM decision:
  Content: 
  Tool calls: [{'name': 'multiply', 'args': {'a': 5847, 'b': 9382}, 'id': 'call_SKSslZrigzoMv4hBPrD4NQKb', 'type': 'tool_call'}]

Step 3 — Tool 'multiply' executed: {'a': 5847, 'b': 9382} = 54856554

Step 4 — Final answer: The result of multiplying 5847 by 9382 is 54,856,554.


In [23]:
def run_agent(user_input, max_iterations=5):
    """Run an agent loop until LLM gives a final answer."""
    
    messages = [
        SystemMessage(content="You are a helpful math assistant. "
    "Use tools when needed. "
    "IMPORTANT: When tasks require sequential operations where one result feeds into the next, "
    "ONLY call ONE tool at a time. Wait for the result before calling the next tool. "
    "Do NOT guess intermediate results."),
        HumanMessage(content=user_input)
    ]
    
    iteration = 0
    
    while iteration < max_iterations:
        iteration += 1
        print(f"\n--- Iteration {iteration} ---")
        
        # LLM decides
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        # Check: tool call or final answer?
        if not response.tool_calls:
            # No tool call → final answer
            print(f"✅ Final answer: {response.content}")
            return response.content
        
        # LLM wants tools — execute them
        print(f"🔧 LLM wants to call {len(response.tool_calls)} tool(s)")
        
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_id = tool_call["id"]
            
            result = multiply.invoke(tool_args)
            print(f"   → {tool_name}({tool_args}) = {result}")
            
            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tool_id
            ))
    
    print("⚠️ Max iterations reached")
    return None

In [24]:
run_agent("What is 5847 × 9382?")


--- Iteration 1 ---
🔧 LLM wants to call 1 tool(s)
   → multiply({'a': 5847, 'b': 9382}) = 54856554

--- Iteration 2 ---
✅ Final answer: The result of multiplying 5847 by 9382 is 54,856,554.


'The result of multiplying 5847 by 9382 is 54,856,554.'

In [25]:
run_agent("Multiply 5 by 3, then multiply that result by 10.")


--- Iteration 1 ---
🔧 LLM wants to call 1 tool(s)
   → multiply({'a': 5, 'b': 3}) = 15

--- Iteration 2 ---
🔧 LLM wants to call 1 tool(s)
   → multiply({'a': 15, 'b': 10}) = 150

--- Iteration 3 ---
✅ Final answer: The result of multiplying 5 by 3 is 15. Multiplying 15 by 10 gives us 150.


'The result of multiplying 5 by 3 is 15. Multiplying 15 by 10 gives us 150.'

In [26]:
run_agent("Multiply 2 by 3, then by 4, then by 5.")


--- Iteration 1 ---
🔧 LLM wants to call 3 tool(s)
   → multiply({'a': 2, 'b': 3}) = 6
   → multiply({'a': 6, 'b': 4}) = 24
   → multiply({'a': 24, 'b': 5}) = 120

--- Iteration 2 ---
✅ Final answer: The result of multiplying 2 by 3 is 6, then multiplying by 4 gives 24, and finally multiplying by 5 gives 120.


'The result of multiplying 2 by 3 is 6, then multiplying by 4 gives 24, and finally multiplying by 5 gives 120.'

In [21]:
run_agent("Hello! How are you?")


--- Iteration 1 ---
✅ Final answer: Hello! I'm here and ready to help you with any math-related questions or problems you have. What can I assist you with today?


"Hello! I'm here and ready to help you with any math-related questions or problems you have. What can I assist you with today?"

In [27]:
run_agent("Multiply 5847 by 9382, then multiply that result by 10.")


--- Iteration 1 ---
🔧 LLM wants to call 1 tool(s)
   → multiply({'a': 5847, 'b': 9382}) = 54856554

--- Iteration 2 ---
🔧 LLM wants to call 1 tool(s)
   → multiply({'a': 54856554, 'b': 10}) = 548565540

--- Iteration 3 ---
✅ Final answer: The result of multiplying 5847 by 9382, and then multiplying that result by 10 is 548565540.


'The result of multiplying 5847 by 9382, and then multiplying that result by 10 is 548565540.'